In [4]:
import numpy as np
import pandas as pd

np.random.seed(73)

wards = ["ICU", "Cardiology", "Respiratory", "General"]
patients = [f"P{i:03d}" for i in range(1, 41)]

dates = pd.date_range("2025-04-01", "2025-04-07", freq="D")

rows = []

for patient in patients:
    ward = np.random.choice(wards)

    for date in dates:
        rows.append({
            "patient_id": patient,
            "ward": ward,
            "date": date,
            "heart_rate": np.random.normal(82, 15),
            "oxygen_sat": np.random.normal(96, 3),
            "temperature": np.random.normal(37.0, 0.6),
            "respiratory_rate": np.random.normal(18, 4)
        })

patients_df = pd.DataFrame(rows)

# Add some missing measurements
for column, n in [
    ("heart_rate", 15),
    ("oxygen_sat", 12),
    ("temperature", 10)
]:
    idx = np.random.choice(
        patients_df.index,
        size=n,
        replace=False
    )
    patients_df.loc[idx, column] = np.nan

patients_df.head()

,patient_id,ward,date,heart_rate,oxygen_sat,temperature,respiratory_rate
0,P001,Respiratory,2025-04-01,95.082784,91.278880,37.044641,12.798176
1,P001,Respiratory,2025-04-02,73.718115,95.237290,36.960054,22.444110
2,P001,Respiratory,2025-04-03,78.214640,103.809746,36.770175,21.186548
3,P001,Respiratory,2025-04-04,76.400928,96.754606,36.108009,16.797864
4,P001,Respiratory,2025-04-05,89.294356,94.397754,36.800266,15.956825


In [10]:
#replace null values with column mean for each
patients_df.fillna(value={
    'heart_rate': patients_df['heart_rate'].mean(),
    'oxygen_sat': patients_df['oxygen_sat'].mean(),
    'temperature': patients_df['temperature'].mean()
},inplace=True
).head()

,patient_id,ward,date,heart_rate,oxygen_sat,temperature,respiratory_rate
0,P001,Respiratory,2025-04-01,95.082784,91.278880,37.044641,12.798176
1,P001,Respiratory,2025-04-02,73.718115,95.237290,36.960054,22.444110
2,P001,Respiratory,2025-04-03,78.214640,103.809746,36.770175,21.186548
3,P001,Respiratory,2025-04-04,76.400928,96.754606,36.108009,16.797864
4,P001,Respiratory,2025-04-05,89.294356,94.397754,36.800266,15.956825


In [43]:
# task 1: ward level patient monitoring
ward_level = patients_df.groupby('ward').agg(
    mean_heart_rate = ('heart_rate','mean'),
    mean_ox_sat = ('oxygen_sat','mean'),
    mean_tenperature = ('temperature','mean'),
    median_res_rate = ('respiratory_rate','median')
)
ward_level['no_uni_patient'] = patients_df.groupby('ward')['patient_id'].unique().count()
ward_level['risk_score'] = ward_level.to_numpy().sum(axis=1)
ward_level.sort_values(by=['mean_heart_rate','mean_ox_sat','mean_tenperature'],ascending=[False,True,False])

,mean_heart_rate,mean_ox_sat,mean_tenperature,median_res_rate,no_uni_patient,risk_score
ward,,,,,,
Respiratory,84.223878,96.311706,36.948453,18.050677,4,239.534714
ICU,81.418643,95.963615,36.956973,17.800513,4,236.139744
Cardiology,81.108047,95.586911,37.036686,18.192549,4,235.924192
General,79.954821,95.782820,36.892197,16.934661,4,233.564499


##### In this dataset Respiratory ward is showing most concerning ward in this hospital as it has patients of having relatively high mean_heart_rate (84.22), low mean_ox_sat (96.31) and high mean_temperature (36.95). And also has high risk_score which is 239.53.

In [ ]:
# task 2: patient deterioration over time
patient_min = patients_df.where(patients_df['date'] == '2025-04-01').dropna()
patient_min
patient_max = patients_df.where(patients_df['date'] == '2025-04-07').dropna()
patient_max.head()

,patient_id,ward,date,heart_rate,oxygen_sat,temperature,respiratory_rate
6,P001,Respiratory,2025-04-07,79.417743,97.592535,37.033886,14.762298
13,P002,Cardiology,2025-04-07,89.098297,96.262903,37.076872,18.696062
20,P003,General,2025-04-07,125.646079,97.331131,37.548879,12.880187
27,P004,General,2025-04-07,81.958943,93.984831,36.610520,17.473959
34,P005,Cardiology,2025-04-07,73.845882,92.845516,36.355575,19.740305


In [41]:
patient_merge = pd.merge(patient_min,patient_max,on='patient_id')
patient_merge.head().drop('ward_y',axis=1)

,patient_id,ward_x,date_x,heart_rate_x,oxygen_sat_x,temperature_x,respiratory_rate_x,date_y,heart_rate_y,oxygen_sat_y,temperature_y,respiratory_rate_y
0,P001,Respiratory,2025-04-01,95.082784,91.278880,37.044641,12.798176,2025-04-07,79.417743,97.592535,37.033886,14.762298
1,P002,Cardiology,2025-04-01,75.063037,96.484420,37.613687,23.657797,2025-04-07,89.098297,96.262903,37.076872,18.696062
2,P003,General,2025-04-01,87.892150,98.872232,36.260408,13.435839,2025-04-07,125.646079,97.331131,37.548879,12.880187
3,P004,General,2025-04-01,81.885095,95.083981,36.723601,13.714449,2025-04-07,81.958943,93.984831,36.610520,17.473959
4,P005,Cardiology,2025-04-01,46.599575,95.717934,37.468669,20.441417,2025-04-07,73.845882,92.845516,36.355575,19.740305


In [57]:
patient_level = patient_merge[['patient_id','ward_x','oxygen_sat_x','oxygen_sat_y']]
patient_level['ox_det'] = patient_level['oxygen_sat_y']-patient_level['oxygen_sat_x']
patient_level.sort_values(by='ox_det',ascending=True).head()

,patient_id,ward_x,oxygen_sat_x,oxygen_sat_y,ox_det
34,P035,Respiratory,93.929414,88.188268,-5.741146
37,P038,Cardiology,96.305612,91.081218,-5.224394
15,P016,Cardiology,94.182767,90.213885,-3.968883
36,P037,Cardiology,96.615560,93.052930,-3.562631
22,P023,Respiratory,95.921387,92.756416,-3.164971


In [ ]:
patients_df.groupby('patient_id').apply(lambda df: pd.Series({
    'day1_ox_sat': df['oxygen_sat'].where(patients_df['date'] == patients_df['date'].min()).dropna(),
    'day7_ox_sat': df['oxygen_sat'].where(patients_df['date'] == patients_df['date'].max()).dropna()
}))

##### Patients having id P035, P038, P016, P037 and P023 are experienced relatively higher decline in their oxygen saturation from first day to last day in the hospital. Most vulnerable patient according to a sharp decline in oxygen saturation is patient P035, having 5.74 downfall in oxygen saturation.

In [72]:
# task 3: heart rate vs oxygen saturation
patients_df[['heart_rate','oxygen_sat']].corr()
patients_df['risk_class'] = np.where(
    (
        (patients_df['heart_rate']>100).astype(int)+
        (patients_df['oxygen_sat']<94).astype(int)
    ) ==2,
    'risk',
    'normal'
)
risk_df = patients_df.groupby('ward').agg(
    hr_over100 = ('heart_rate',lambda x: (x>100).sum()),
    os_below94 = ('oxygen_sat', lambda x: (x<94).sum()),
    risk_count = ('risk_class', lambda x: (x=="risk").sum()),
    total_obs = ('risk_class','size')
)
risk_df['risk_perc'] = (risk_df['risk_count']/risk_df['total_obs'])*100
risk_df

,hr_over100,os_below94,risk_count,total_obs,risk_perc
ward,,,,,
Cardiology,7,19,1,77,1.298701
General,4,8,0,42,0.000000
ICU,7,26,1,84,1.190476
Respiratory,12,19,2,77,2.597403


##### Heart rate and oxygen saturation are not correlated to each other as their correlation value is 0.15. In this dataset respiratory ward has high risk patients as it has 2 high risk patients of total 4.